In [1]:
import pandas as pd
file = "C:\\Users\\eyasf\\OneDrive\\Desktop\\projet_raies\\raw_data\\FX Data.xlsx"

xls = pd.ExcelFile(file)
print(xls.sheet_names)


['InterBancaire', 'BCT Fixing EUR', 'BCT Fixing USD', 'EUR-GBP-JPY vs USD']


In [ ]:
#importing the data
Bct_fixing_USD = pd.read_excel(xls, "BCT Fixing USD")
Interbancaire = pd.read_excel(xls, "InterBancaire")
print(Bct_fixing_USD.head())
print(Interbancaire.head())

  Exchange Date     Bid     Ask  Mid Price
0    2020-04-10  2.9049  2.9079    2.90640
1    2020-04-13  2.8954  2.8982    2.89680
2    2020-04-14  2.8877  2.8905    2.88910
3    2020-04-15  2.8938  2.8966    2.89520
4    2020-04-16  2.8999  2.9028    2.90135
         Date     DZD     SAR     CAD      DKK     USD    GBP      JPY  \
0  2004-01-01  0.1781  3.3988  0.9138  20.5241  1.2741  2.257  11.2003   
1  2004-01-02  0.1781  3.3988  0.9138  20.5241  1.2741  2.257  11.2003   
2  2004-01-03  0.1781  3.3988  0.9138  20.5241  1.2741  2.257  11.2003   
3  2004-01-04  0.1781  3.3988  0.9138  20.5241  1.2741  2.257  11.2003   
4  2004-01-05  0.1781  3.3988  0.9138  20.5241  1.2741  2.257  11.2003   

      MAD      NOK  ...     CHF     KWD     AED     EUR  Unnamed: 15  \
0  1.3957  18.4491  ...  9.9535  4.3237  3.4717  1.5271          NaN   
1  1.3957  18.4491  ...  9.9535  4.3237  3.4717  1.5271          NaN   
2  1.3957  18.4491  ...  9.9535  4.3237  3.4717  1.5271          NaN   
3  1.3957

In [6]:
# cleaning the interbank 
IB = Interbancaire[["Date", "USD"]]
IB = IB.rename(columns={"USD": "USD_IB"})
IB['Date'] = pd.to_datetime(IB['Date'])
IB.head()


,Date,USD_IB
0,2004-01-01,1.2741
1,2004-01-02,1.2741
2,2004-01-03,1.2741
3,2004-01-04,1.2741
4,2004-01-05,1.2741


In [9]:
# clean fixing 
Fix = Bct_fixing_USD[['Exchange Date','Mid Price']]
Fix.columns = ['Date','Fixing_USD']
Fix['Date'] = pd.to_datetime(Fix['Date'])
Fix.head()


C:\Users\eyasf\AppData\Local\Temp\ipykernel_21400\913866407.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Fix['Date'] = pd.to_datetime(Fix['Date'])


,Date,Fixing_USD
0,2020-04-10,2.90640
1,2020-04-13,2.89680
2,2020-04-14,2.88910
3,2020-04-15,2.89520
4,2020-04-16,2.90135


In [11]:
IB = IB[IB['Date'] >= Fix['Date'].min()]
IB.head()

,Date,USD_IB
5944,2020-04-10,2.9107
5945,2020-04-11,2.9107
5946,2020-04-12,2.9107
5947,2020-04-13,2.8924
5948,2020-04-14,2.8894


In [13]:
#merge the two datasets
df = pd.merge(IB, Fix, on='Date', how='inner')
df.head()

,Date,USD_IB,Fixing_USD
0,2020-04-10,2.9107,2.90640
1,2020-04-13,2.8924,2.89680
2,2020-04-14,2.8894,2.88910
3,2020-04-15,2.8874,2.89520
4,2020-04-16,2.9006,2.90135


In [15]:
df['Fixing_lag'] = df['Fixing_USD'].shift(1)
df.head()

,Date,USD_IB,Fixing_USD,Fixing_lag
0,2020-04-10,2.9107,2.90640,NaN
1,2020-04-13,2.8924,2.89680,2.9064
2,2020-04-14,2.8894,2.88910,2.8968
3,2020-04-15,2.8874,2.89520,2.8891
4,2020-04-16,2.9006,2.90135,2.8952


In [18]:
#spread
df['Spread_same'] = df['USD_IB'] - df['Fixing_USD']
df['Spread_lag'] = df['USD_IB'] - df['Fixing_lag']
df.head()

,Date,USD_IB,Fixing_USD,Fixing_lag,Spread_same,Spread_lag
0,2020-04-10,2.9107,2.90640,NaN,0.00430,NaN
1,2020-04-13,2.8924,2.89680,2.9064,-0.00440,-0.0140
2,2020-04-14,2.8894,2.88910,2.8968,0.00030,-0.0074
3,2020-04-15,2.8874,2.89520,2.8891,-0.00780,-0.0017
4,2020-04-16,2.9006,2.90135,2.8952,-0.00075,0.0054


In [19]:
df = df.dropna()